# 05 — Split and baseline

## Goal of this step

Understand the fixed stratified 80/20 split used since EDA. Measure two reference models using five-fold CV on training customers only.

## Why this matters

Stratification is like dealing cards so both hands contain a similar share of rare cards. CV rotates practice exams inside the training set; the held-out test is a sealed final exam.

Find the project root and import display tools. Reusable logic lives in src.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np  # noqa: F401
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
plt.rcParams["figure.figsize"] = (8, 4)


Use the shared split with random_state=42. Stratification preserves the minority-class share; 5,616 customers train and 1,405 remain held out.

In [2]:
from src.split import split_data
from src.train import run_baselines, comparison_display
x_train, x_test, y_train, y_test = split_data()
display(pd.Series({'train_rows': len(x_train), 'test_rows': len(x_test), 'train_churn_share': y_train.mean(), 'index_overlap': len(set(x_train.index) & set(x_test.index))}))

train_rows           5616.000000
test_rows            1405.000000
train_churn_share       0.264423
index_overlap           0.000000
dtype: float64

**Interpretation:** The partitions do not overlap. The test labels are reserved for notebook 07 and are never passed to fit or tuning.

Both models include feature creation and preprocessing inside a Pipeline. The dummy always predicts the majority; balanced logistic regression gives minority errors more weight. CV fits each preprocessor on only four folds, then scores on the fifth.

In [3]:
baselines = run_baselines()
display(comparison_display(baselines))

,roc_auc,average_precision,recall,precision,f1,accuracy
model,,,,,,
Dummy,0.5000 +/- 0.0000,0.2644 +/- 0.0001,0.0000 +/- 0.0000,0.0000 +/- 0.0000,0.0000 +/- 0.0000,0.7356 +/- 0.0001
LogisticRegression,0.8465 +/- 0.0195,0.6603 +/- 0.0378,0.8034 +/- 0.0340,0.5182 +/- 0.0181,0.6300 +/- 0.0229,0.7505 +/- 0.0149


**Interpretation:** ROC-AUC measures ranking across both classes; PR-AUC here means average precision, which focuses on finding churners. Recall measures coverage, precision offer efficiency, and F1 their harmonic balance.

Show why accuracy is misleading. A model can win many easy majority cases while failing its actual business purpose.

In [4]:
dummy = baselines.loc[baselines['model'].eq('Dummy')].iloc[0]
display(pd.Series({'dummy_CV_accuracy': dummy['accuracy_mean'], 'dummy_CV_recall': dummy['recall_mean'], 'dummy_CV_PR_AUC_AP': dummy['average_precision_mean']}))

dummy_CV_accuracy     0.735577
dummy_CV_recall       0.000000
dummy_CV_PR_AUC_AP    0.264423
dtype: float64

**Interpretation:** The dummy has about 73.6% accuracy but zero churn recall. Its average precision equals churn prevalence; all customers receive the same zero risk score.

## Common mistakes

- Comparing models using different folds.
- Fitting encoding before CV.
- Treating high majority-class accuracy as churn detection success.

## Key takeaways

- The 20% test set is untouched by model decisions.
- All preprocessing is fit inside training folds.
- The logistic baseline must improve on a meaningful dummy reference.

## Next step

06_model_training.ipynb: compare three models using identical folds and tune the best by average precision.